# Scraping Sofascore

En este cuaderno se prueban las funciones para recopilar los resultados de los jugadores de Sofascore.

## Dependencias

In [41]:
import pandas as pd
import numpy as np
import requests
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

import os 
import os.path as osp

import soccerdata as sd

In [16]:
url_lineup = "https://www.sofascore.com/api/v1/event/12501501/lineups"

response = requests.get(url_lineup)
data = response.json()

In [60]:
def extract_player_stats(players_data, team_side: str):
    """
    Extract player statistics from team players data
    
    Args:
        players_data (list): List of player dictionaries from team data
        team_side (str): 'home' or 'away' to identify team side
        
    Returns:
        list: List of dictionaries with extracted player stats
    """
    extracted_stats = []
    
    for player_data in players_data:
        # Extract basic player info
        player_info = player_data['player']
        stats = player_data.get('statistics', {})
        
        # Start with basic player info
        player_stats = {
            'id': player_info['id'],
            'name': player_info['name'],
            'position': player_data['position'],
            'shirt_number': player_data['shirtNumber'],
            'team_id': player_data['teamId'],
            'team_side': team_side,
            'substitute': player_data.get('substitute', False),
            'captain': player_data.get('captain', False)
        }
        
        # Dynamically add all available statistics
        for stat_key, stat_value in stats.items():
            if stat_key == 'ratingVersions' and isinstance(stat_value, dict):
                # Handle nested rating versions
                for version_key, version_value in stat_value.items():
                    player_stats[f'rating_{version_key}'] = version_value
            else:
                # Convert camelCase to snake_case for consistency
                snake_case_key = ''.join(['_' + c.lower() if c.isupper() else c for c in stat_key]).lstrip('_')
                player_stats[snake_case_key] = stat_value
        
        extracted_stats.append(player_stats)
    
    return extracted_stats

In [58]:
from tqdm import tqdm
import time

def extract_match_players_stats(match_ids, base_url: str = "https://api.sofascore.com/api/v1/"):
    """
    Extract player statistics for multiple matches
    
    Args:
        match_ids (List[int]): List of match IDs to process
        base_url (str): Base URL for the API
        
    Returns:
        Dict[int, List[Dict]]: Dictionary with match_id as key and list of player stats as value
    """
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    all_matches_stats = {}
    
    for match_id in match_ids:
        try:
            # Get match statistics
            match_url = f"{base_url}event/{match_id}/lineups"
            response = requests.get(match_url, headers=headers)
            response.raise_for_status()
            
            match_data = response.json()
            
            # Extract player statistics from both home and away teams
            all_players_stats = []
            
            # Process home team players
            if 'home' in match_data and 'players' in match_data['home']:
                home_players = extract_player_stats(match_data['home']['players'], 'home')
                all_players_stats.extend(home_players)
            
            # Process away team players
            if 'away' in match_data and 'players' in match_data['away']:
                away_players = extract_player_stats(match_data['away']['players'], 'away')
                all_players_stats.extend(away_players)
            
            all_matches_stats[match_id] = all_players_stats
            
            print(f"✓ Extracted stats for match {match_id} - {len(all_players_stats)} players")
            
            # Rate limiting - be respectful to the API
            time.sleep(1)
            
        except requests.exceptions.RequestException as e:
            print(f"✗ Error fetching match {match_id}: {e}")
            all_matches_stats[match_id] = []
            
        except Exception as e:
            print(f"✗ Unexpected error processing match {match_id}: {e}")
            all_matches_stats[match_id] = []
    
    return all_matches_stats
    

In [24]:
import soccerdata as sd


sd_sofascore = sd.Sofascore(leagues=['ITA-Serie A'], seasons=['2024-2025']) 

[08/12/25 17:27:29] INFO     Saving cached data to C:\Users\edoar\soccerdata\data\Sofascore          ]8;id=4901;file://e:\Edoardo\Education\SportDataCampus\SportsDataCampus\.venv\lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=198595;file://e:\Edoardo\Education\SportDataCampus\SportsDataCampus\.venv\lib\site-packages\soccerdata\_common.py#263\263]8;;\

In [ ]:
SOFASCORE_API = "https://api.sofascore.com/api/v1/"
competitionId = 23
seasonId = 63515
urlmask1 = SOFASCORE_API + f"unique-tournament/{competitionId}/season/{seasonId}/rounds"
urlmask2 = SOFASCORE_API + f"unique-tournament/{competitionId}/season/{seasonId}/events/round/1"

requests.get(urlmask2).json()['events'][0]

In [43]:
import requests
import time
from typing import List, Dict, Any

def extract_round_events(competition_id: int, season_id: int, base_url: str = "https://api.sofascore.com/api/v1/") -> Dict[int, List[Dict[str, Any]]]:
    """
    Extract all events for each round of a competition season
    
    Args:
        competition_id (int): Competition ID
        season_id (int): Season ID
        base_url (str): Base URL for the API
        
    Returns:
        Dict[int, List[Dict]]: Dictionary with round number as key and list of events as value
    """
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    # Get all rounds for the season
    rounds_url = f"{base_url}unique-tournament/{competition_id}/season/{season_id}/rounds"
    
    try:
        rounds_response = requests.get(rounds_url, headers=headers)
        rounds_response.raise_for_status()
        rounds_data = rounds_response.json()
        
        all_rounds_events = []
        
        for round_info in rounds_data.get('rounds', []):
            round_number = round_info.get('round')
            
            # Get events for this specific round
            events_url = f"{base_url}unique-tournament/{competition_id}/season/{season_id}/events/round/{round_number}"
            
            try:
                events_response = requests.get(events_url, headers=headers)
                events_response.raise_for_status()
                events_data = events_response.json()
                
                # Extract relevant event information
                round_events = []
                for event in events_data.get('events', []):
                    event_info = {
                        'round': round_number,
                        'competition_name': event['tournament']['name'],
                        'competition_id': event['tournament']['id'],
                        'season_name': event['season']['name'],
                        'season_id': event['season']['id'],
                        'home_team_name': event['homeTeam']['name'],
                        'home_team_id': event['homeTeam']['id'],
                        'away_team_name': event['awayTeam']['name'],
                        'away_team_id': event['awayTeam']['id'],
                        'match_id': event['id'],
                        'status': event['status']['description'],
                        'start_timestamp': event.get('startTimestamp'),
                        'home_score': event['homeScore'].get('current', 0),
                        'away_score': event['awayScore'].get('current', 0)
                    }
                    all_rounds_events.append(event_info)
                
                
                print(f"✓ Round {round_number}: {len(events_data)} events")
                
                # Rate limiting
                time.sleep(0.5)
                
            except requests.exceptions.RequestException as e:
                print(f"✗ Error fetching round {round_number}: {e}")
                all_rounds_events[round_number] = []
                
    except requests.exceptions.RequestException as e:
        print(f"✗ Error fetching rounds: {e}")
        return {}
    
    return all_rounds_events

# Example usage:
competition_id = 23  # Serie A
season_id = 63515   # 24/25 season
rounds_events = extract_round_events(competition_id, season_id)

# Access events for a specific round:
# round_1_events = rounds_events[1]
# print(f"Round 1 has {len(round_1_events)} matches")

# Process all rounds:
# for round_num, events in rounds_events.items():
#     print(f"\nRound {round_num}:")
#     for event in events:
#         print(f"  {event['home_team_name']} vs {event['away_team_name']} ({event['home_score']}-{event['away_score']})")

# Get all match IDs for further processing:
# all_match_ids = []
# for round_events in rounds_events.values():
#     for event in round_events:
#         all_match_ids.append(event['match_id'])
# print(f"Total matches: {len(all_match_ids)}")

✓ Round 1: 0 events
✓ Round 2: 0 events
✓ Round 3: 0 events
✓ Round 4: 0 events
✓ Round 5: 0 events
✓ Round 6: 0 events
✓ Round 7: 0 events
✓ Round 8: 0 events
✓ Round 9: 0 events
✓ Round 10: 0 events
✓ Round 11: 0 events
✓ Round 12: 0 events
✓ Round 13: 0 events
✓ Round 14: 0 events
✓ Round 15: 0 events
✓ Round 16: 0 events
✓ Round 17: 0 events
✓ Round 18: 0 events
✓ Round 19: 0 events
✓ Round 20: 0 events
✓ Round 21: 0 events
✓ Round 22: 0 events
✓ Round 23: 0 events
✓ Round 24: 0 events
✓ Round 25: 0 events
✓ Round 26: 0 events
✓ Round 27: 0 events
✓ Round 28: 0 events
✓ Round 29: 0 events
✓ Round 30: 0 events
✓ Round 31: 0 events
✓ Round 32: 0 events
✓ Round 33: 0 events
✓ Round 34: 0 events
✓ Round 35: 0 events
✓ Round 36: 0 events
✓ Round 37: 0 events
✓ Round 38: 0 events


In [45]:
schedule = pd.json_normalize(rounds_events)

In [ ]:
ids = schedule['match_id'].tolist()


res = extract_match_players_stats(ids)



In [70]:
def save_players_stats_to_csv(all_matches_stats: Dict[int, List[Dict[str, Any]]], 
                             competition_name: str, 
                             season_name: str,
                             notebook_path: str = None) -> str:
    """
    Convert match players stats to DataFrame and save to CSV
    
    Args:
        all_matches_stats (Dict): Output from extract_match_players_stats
        competition_name (str): Name of the competition (e.g., 'Serie A')
        season_name (str): Name of the season (e.g., '24/25')
        notebook_path (str): Path to the notebook (optional, auto-detected if None)
        
    Returns:
        str: Path to the saved CSV file
    """
    
    # Auto-detect notebook path if not provided
    if notebook_path is None:
        notebook_path = os.getcwd()
    
    # Get parent folder path (where the 'data' folder should be)
    parent_folder = os.path.dirname(notebook_path)
    data_folder = os.path.join(parent_folder, 'data')
    
    # Create data folder if it doesn't exist
    os.makedirs(data_folder, exist_ok=True)
    
    # Convert to DataFrame
    all_players_list = []
    
    for match_id, players in all_matches_stats.items():
        for player in players:
            # Add match_id to each player record
            player_with_match = player.copy()
            player_with_match['match_id'] = match_id
            all_players_list.append(player_with_match)
    
    # Create DataFrame
    df = pd.DataFrame(all_players_list)
    
    # Generate filename
    filename = f"{competition_name}_{season_name}.csv"
    filepath = os.path.join(data_folder, filename)
    
    # Save to CSV
    df.to_csv(filepath, index=False, encoding='utf-8')
    
    print(f"✓ Saved {len(df)} player records to: {filepath}")
    print(f"  DataFrame shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    return filepath

In [71]:
save_players_stats_to_csv(res, competition_name = 'SerieA', season_name='2024_2025',)

✓ Saved 17704 player records to: e:\Edoardo\Education\SportDataCampus\SportsDataCampus\Module4\tarea_individual\data\SerieA_2024_2025.csv
  DataFrame shape: (17704, 63)
  Columns: ['id', 'name', 'position', 'shirt_number', 'team_id', 'team_side', 'substitute', 'captain', 'total_pass', 'accurate_pass', 'total_long_balls', 'accurate_long_balls', 'goal_assist', 'good_high_claim', 'saved_shots_from_inside_the_box', 'saves', 'minutes_played', 'touches', 'rating', 'possession_lost_ctrl', 'rating_original', 'rating_alternative', 'goals_prevented', 'match_id', 'total_cross', 'accurate_cross', 'aerial_lost', 'duel_lost', 'duel_won', 'dispossessed', 'on_target_scoring_attempt', 'goals', 'total_clearance', 'outfielder_block', 'interception_won', 'total_tackle', 'fouls', 'expected_goals', 'expected_assists', 'aerial_won', 'was_fouled', 'shot_off_target', 'challenge_lost', 'total_contest', 'won_contest', 'big_chance_created', 'clearance_off_line', 'key_pass', 'error_lead_to_a_shot', 'big_chance_mis

'e:\\Edoardo\\Education\\SportDataCampus\\SportsDataCampus\\Module4\\tarea_individual\\data\\SerieA_2024_2025.csv'